In [0]:

from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as spark_sum,
    avg, round as spark_round,
    max as spark_max, min as spark_min,
    countDistinct, when, lit
)

spark = SparkSession.builder.getOrCreate()
spark.sql("USE fraud_db")

silver_df = spark.table("fraud_db.silver_transactions")

print(f"✅ Silver records loaded : {silver_df.count():,}")
print(f"   Columns available     : {len(silver_df.columns)}")

In [0]:

gold_by_country = silver_df \
    .groupBy("country_code") \
    .agg(
        count("*")                              .alias("total_txns"),
        spark_sum("is_fraud")                   .alias("fraud_txns"),
        spark_round(avg("amount"), 2)           .alias("avg_amount"),
        spark_round(spark_sum("amount"), 2)     .alias("total_amount"),
        spark_round(
            spark_sum("is_fraud") / count("*") * 100, 2
        )                                       .alias("fraud_rate_pct")
    ) \
    .orderBy("fraud_rate_pct", ascending=False)

spark.sql("DROP TABLE IF EXISTS fraud_db.gold_fraud_by_country")

gold_by_country.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fraud_db.gold_fraud_by_country")

print("✅ Gold Table 1: fraud_by_country")
print("   🌍 Fraud Rate by Country:")
display(spark.table("fraud_db.gold_fraud_by_country"))

In [0]:
gold_by_merchant = silver_df \
    .groupBy("merchant_category") \
    .agg(
        count("*")                              .alias("total_txns"),
        spark_sum("is_fraud")                   .alias("fraud_txns"),
        spark_round(avg("amount"), 2)           .alias("avg_txn_amount"),
        spark_round(
            spark_sum("is_fraud") / count("*") * 100, 2
        )                                       .alias("fraud_rate_pct"),
        spark_round(
            spark_sum(
                when(col("is_fraud") == 1, col("amount")).otherwise(0)
            ), 2
        )                                       .alias("total_fraud_amount")
    ) \
    .orderBy("fraud_rate_pct", ascending=False)

spark.sql("DROP TABLE IF EXISTS fraud_db.gold_fraud_by_merchant")

gold_by_merchant.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fraud_db.gold_fraud_by_merchant")

print("✅ Gold Table 2: fraud_by_merchant")
print("   🏪 Fraud Rate by Merchant Category:")
display(spark.table("fraud_db.gold_fraud_by_merchant"))

In [0]:
gold_by_hour = silver_df \
    .groupBy("hour_of_day") \
    .agg(
        count("*")                              .alias("total_txns"),
        spark_sum("is_fraud")                   .alias("fraud_txns"),
        spark_round(
            spark_sum("is_fraud") / count("*") * 100, 2
        )                                       .alias("fraud_rate_pct"),
        spark_round(avg("amount"), 2)           .alias("avg_amount")
    ) \
    .orderBy("hour_of_day")

spark.sql("DROP TABLE IF EXISTS fraud_db.gold_fraud_by_hour")

gold_by_hour.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fraud_db.gold_fraud_by_hour")

print("✅ Gold Table 3: fraud_by_hour")
print("   ⏰ Fraud Rate Across 24 Hours:")
display(spark.table("fraud_db.gold_fraud_by_hour"))

In [0]:

from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg,
    round as spark_round, max as spark_max, when, lit
)

silver_with_score = silver_df.withColumn(
    "risk_severity",
    when(col("risk_label") == "CRITICAL", lit(5))
    .when(col("risk_label") == "HIGH",    lit(4))
    .when(col("risk_label") == "MEDIUM",  lit(3))
    .when(col("risk_label") == "LOW",     lit(2))
    .otherwise(lit(1))
)

gold_risky_customers = silver_with_score \
    .groupBy("customer_id") \
    .agg(
        count("*")                              .alias("total_txns"),
        spark_sum("is_fraud")                   .alias("fraud_txns"),
        spark_round(spark_sum("amount"), 2)     .alias("total_spent"),
        spark_round(
            spark_sum("is_fraud") / count("*") * 100, 2
        )                                       .alias("fraud_rate_pct"),
        spark_round(avg("fraud_risk_score"), 2) .alias("avg_risk_score"),
        spark_max("risk_severity")              .alias("max_severity")
    ) \
    .filter(col("fraud_txns") > 0) \
    .withColumn(
        "highest_risk_label",
        when(col("max_severity") == 5, lit("CRITICAL"))
        .when(col("max_severity") == 4, lit("HIGH"))
        .when(col("max_severity") == 3, lit("MEDIUM"))
        .when(col("max_severity") == 2, lit("LOW"))
        .otherwise(lit("SAFE"))
    ) \
    .drop("max_severity") \
    .orderBy("fraud_txns", ascending=False)

spark.sql("DROP TABLE IF EXISTS fraud_db.gold_risky_customers")

gold_risky_customers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fraud_db.gold_risky_customers")

# ← renamed from 'count' to 'cust_count' — avoids overwriting count()
cust_count = spark.table("fraud_db.gold_risky_customers").count()
print(f"✅ Gold Table 4 FIXED: gold_risky_customers")
print(f"   👤 High-risk customers : {cust_count:,}")
print(f"\n   Top 10 Riskiest Customers:")
display(spark.table("fraud_db.gold_risky_customers").limit(10))

In [0]:
# ============================================================
# 03_Gold_Layer | CELL 6 — Executive KPI Dashboard Table
# ============================================================
# WHAT  : One single-row summary of ALL fraud KPIs.
# WHY   : The CEO/Head of Risk doesn't read 1,500 rows.
#         They read ONE number: "What's our fraud rate today?"
#         This table is what powers the TOP of every
#         fraud dashboard — the headline metrics.
# ============================================================

from pyspark.sql.functions import spark_partition_id

total_txns    = silver_df.count()
fraud_txns    = silver_df.filter(col("is_fraud") == 1).count()
legit_txns    = total_txns - fraud_txns
total_amount  = silver_df.agg(
                    spark_round(spark_sum("amount"), 2)
                ).collect()[0][0]
fraud_amount  = silver_df.filter(col("is_fraud") == 1).agg(
                    spark_round(spark_sum("amount"), 2)
                ).collect()[0][0]
avg_risk      = silver_df.agg(
                    spark_round(avg("fraud_risk_score"), 2)
                ).collect()[0][0]
critical_txns = silver_df.filter(col("risk_label") == "CRITICAL").count()

# Build summary as a single-row DataFrame
summary_data = [(
    total_txns,
    fraud_txns,
    legit_txns,
    round(fraud_txns / total_txns * 100, 2),
    float(total_amount),
    float(fraud_amount),
    round(float(fraud_amount) / float(total_amount) * 100, 2),
    float(avg_risk),
    critical_txns
)]

summary_cols = [
    "total_transactions", "fraud_transactions", "legit_transactions",
    "overall_fraud_rate_pct", "total_txn_amount_usd",
    "total_fraud_amount_usd", "fraud_amount_pct",
    "avg_portfolio_risk_score", "critical_risk_transactions"
]

gold_kpi = spark.createDataFrame(summary_data, summary_cols)

spark.sql("DROP TABLE IF EXISTS fraud_db.gold_kpi_summary")

gold_kpi.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("fraud_db.gold_kpi_summary")

print("✅ Gold Table 5: gold_kpi_summary")
print("\n📊 EXECUTIVE KPI DASHBOARD:")
print("=" * 55)
kpi = spark.table("fraud_db.gold_kpi_summary").collect()[0]
print(f"  💳 Total Transactions     : {kpi['total_transactions']:,}")
print(f"  🚨 Fraud Transactions     : {kpi['fraud_transactions']:,}")
print(f"  📊 Overall Fraud Rate     : {kpi['overall_fraud_rate_pct']}%")
print(f"  💰 Total Amount (USD)     : ${kpi['total_txn_amount_usd']:,.2f}")
print(f"  🔴 Fraud Amount (USD)     : ${kpi['total_fraud_amount_usd']:,.2f}")
print(f"  📉 Fraud Amount %         : {kpi['fraud_amount_pct']}%")
print(f"  ⚠️  Critical Risk Txns     : {kpi['critical_risk_transactions']:,}")
print(f"  🎯 Avg Portfolio Risk     : {kpi['avg_portfolio_risk_score']}/5")
print("=" * 55)

In [0]:
# ============================================================
# 03_Gold_Layer | CELL 7 — Final Validation
# ============================================================

print("🥇 GOLD LAYER — ALL TABLES SUMMARY")
print("=" * 55)

gold_tables = {
    "gold_fraud_by_country"  : "🌍 Fraud by Country",
    "gold_fraud_by_merchant" : "🏪 Fraud by Merchant",
    "gold_fraud_by_hour"     : "⏰ Fraud by Hour",
    "gold_risky_customers"   : "👤 Risky Customers",
    "gold_kpi_summary"       : "📊 Executive KPIs",
}

for table, label in gold_tables.items():
    count = spark.table(f"fraud_db.{table}").count()
    print(f"  ✅ {label:30s} → {count:,} rows")

print("=" * 55)
print("\n🏗️  FULL MEDALLION ARCHITECTURE STATUS:")
print(f"  🪨 Bronze : fraud_db.bronze_transactions   → {spark.table('fraud_db.bronze_transactions').count():,} rows")
print(f"  🥈 Silver : fraud_db.silver_transactions   → {spark.table('fraud_db.silver_transactions').count():,} rows")
print(f"  🥇 Gold   : 5 business tables              → ✅ ALL READY")
print("\n  🟢 Medallion Architecture COMPLETE!")
print("  ⏭️  Next → Phase 4: MLflow + Fraud Detection ML Model")

